In [ ]:
import sys,os,shutil,getpass
# the work dir must be in the root of "Source"
work_dir = os.getcwd()
if 'notebooks' in work_dir :
  parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)
#print(parent_dir)

In [ ]:
# just to hide the warning messages

import warnings
warnings.filterwarnings('ignore')


In [ ]:
import altair as alt
import numpy as np
import subprocess
import folium
import random
import matplotlib.pyplot as plt
import pandas as pd
import json
import math
import numpy as np
import numpy.ma as ma
import datetime

from urllib.parse import urlparse
from folium import plugins
from netCDF4 import Dataset,num2date
from functools import reduce



### Cal/Val module
***
SOURCE evaluates model performance through the computation of basic __skill's scores__ at the platform locations once the observed and synthetic data are generated. The cal/val module loads both observed and synthetic data time series and computes the class 4 metrics (`Simoncelli et al., 2016`). 

#### 1. Choose the North Adriatic Sea climatology to be shown

In [ ]:
# Choose the North Adriatic Sea climatology to be shown.
# Assign one of the two following values to climdataset variable:
# climnadrdaily (INGV climatology) || climuliegedaily (ULiege climatology)
climdataset = 'climuliegedaily'


#### 2. Temperature dataframe creation
***

In order to visualize observations and model outputs, first it has to be created a dataframe containing needed information

In [ ]:
debug = False # Set to False to silence debug output

output_directory = parent_dir+'/output_NA/'

InsMerDir = output_directory+'OBSERVATION/output/statistic/dm/sea_water_temperature' #merged/dm/sea_water_temperature'
ModMerDir = output_directory+'MODEL/output/post-processed/dm/sea_water_temperature' #merged/dm/sea_water_temperature'
ClimMerDir = output_directory+'CLIM/output/statistic/dm/sea_water_temperature' #merged/dm/sea_water_temperature'


######################################
####### INSITU OBSERVATION ###########
######################################

pcode = []
pname = []
wcode = []
t = []
lon = []
lat = []
depth = []
obse = []
k_obs = []

mod_T = []
mod_time = []
k_mod = []

clim_T = []
clim_time = []
k_clim = []


sorted_files_obs = sorted([ x for x in os.listdir(InsMerDir) if x.endswith('nc')])
sorted_files_mod = sorted([ x for x in os.listdir(ModMerDir) if x.endswith('nc')])
sorted_files_clim = sorted([ x for x in os.listdir(ClimMerDir) if x.endswith('nc') and x.startswith(climdataset)])

for f in sorted_files_obs:
    mooring = f.split('_')[1]
    print(f"** Reading Platform: {mooring}")
    model_file = [mod_f for mod_f in sorted_files_mod if mod_f.split('_')[1] == mooring][0]

    fclim = 0
    try:
        clim_file = [clim_f for clim_f in sorted_files_clim if clim_f.split('_')[1] == mooring][0]
    except IndexError as e:
        msg = f"Climatological temperature file not found for platform {mooring}"
        print(msg)
    else:
        msg = f"Climatological temperature file ** FOUND ** for platform {mooring}"
        print(msg)
        fclim = 1        
       
    
    # Read insitu netcdf
    i_nc = Dataset(InsMerDir+'/'+f,'r')
    print(f"Insitu file: {InsMerDir+'/'+f}")
    t.append(i_nc.variables['time'][:])
    lon.append(i_nc.variables['lon'][:])
    lat.append(i_nc.variables['lat'][:])
    depth.append(i_nc.variables['depth'][:])
    obse.append(np.squeeze(i_nc.variables['sea_water_temperature']))
    k_obs.append('insitu')
    pcode.append(i_nc.getncattr('platform_code'))

    # For some probes platform_name is not present or empty.
    # In this cases use platform_code instead
    final_value = None # Initialize a variable for the final value

    # 1. Try to use 'platform_name'
    if hasattr(i_nc, 'platform_name'):
        value = i_nc.getncattr('platform_name')
        # Check that the value is not empty or just whitespace
        if value and str(value).strip():
            final_value = value

    # 2. If no value was found, try with 'platform_code'
    if not final_value and hasattr(i_nc, 'platform_code'):
        code_value = i_nc.getncattr('platform_code')
        # Check here as well that the value is not empty
        if code_value and str(code_value).strip():
            final_value = code_value

    # 3. Finally, append the value to the list only if a valid one was found
    if final_value:
        pname.append(final_value)

    wcode.append(i_nc.getncattr('wmo_platform_code'))

    
#     # ######################################
#     # #######        MODEL       ###########
#     # ######################################

    # Read model netcdf
    m_nc = Dataset(ModMerDir+'/'+model_file,'r')
    print(f"Model file: {ModMerDir+'/'+model_file}")
    mod_time.append(m_nc.variables['time'][:])
    mod_T.append(np.squeeze(m_nc.variables['sea_water_temperature']))
    k_mod.append('model')


#     # ######################################
#     # #########   CLIMATOLOGY    ###########
#     # ######################################

    # Read climatology netcdf
    if fclim == 1:
        c_nc = Dataset(ClimMerDir+'/'+clim_file,'r')
        print(f"Climatology file: {ClimMerDir+'/'+clim_file}")
        clim_time.append(c_nc.variables['time'][:])
        clim_T.append(np.squeeze(c_nc.variables['sea_water_temperature']))
        k_clim.append('climatology')
    elif fclim == 0:
        offset_in_minutes = (datetime.datetime(1970, 1, 1) - datetime.datetime(1900, 1, 1)).total_seconds() / 60
        clim_time_seconds = (m_nc.variables['time'][:] - offset_in_minutes) * 60 + 43200
        clim_time.append(clim_time_seconds)
        clim_T.append([np.nan] * len((m_nc.variables['time'][:])))
        k_clim.append('climatology')

# end of the loop on sorted_files_obs

# Create dataframes
data_obs = pd.DataFrame({'platform_code': pcode,
        'platform_name': pname,
        'wmo_platform_code': wcode,
        'time': t,
        'lon': lon,
        'lat': lat,
        'depth': depth,
        'obs': obse,
        'kind': k_obs
        })

data_mod = pd.DataFrame({'time': mod_time,'model': mod_T,'kind': k_mod})

data_clim = pd.DataFrame({'time': clim_time,'clim': clim_T,'kind': k_clim})

if debug:
    display(data_clim)  # or display(data_obs) or display(data_mod)

#### 3. Visualization of temperature data and computation of model skill scores
***

Temperature time series visualization, RMSE and bias computation

In [ ]:
# --- INITIAL SETUP ---
debug = False  # Set to True to enable debug output
m = folium.Map(location=[40.5, 12.5], zoom_start=6)
icon_url = parent_dir + '/ICON/buoy_icon_2.png'

# --- MAIN LOOP OVER PLATFORMS ---
for p in np.arange(0,data_obs.shape[0]):    
    print(f"** Processing Platform: {data_obs['platform_code'][p]}")
    
    charts_for_platform = []
    lat = data_obs['lat'][p]
    lon = data_obs['lon'][p]
    platform_code = str(data_obs['platform_code'][p])
    # Modify platform_name for the probes where platform_name is not present or empty.
    platform_name = str(data_obs['platform_name'][p])
    if platform_name == '68422':
        platform_name = 'Pylos'
    elif platform_name == '61277':
        platform_name = 'E1M3A'
    elif platform_name == 'HERAKLION':
        platform_name = 'Heraklion'

    # --- INNER LOOP OVER DEPTHS ---
    for i, depth_level in enumerate(data_obs['depth'][p]):
        depth_label = str(np.around(depth_level,2))
        print(f"  - Processing depth: {depth_label} m")

        try:
            # --- DATA PREPARATION FOR THE CURRENT DEPTH ---
            offset_in_minutes = (datetime.datetime(1970, 1, 1) - datetime.datetime(1900, 1, 1)).total_seconds() / 60
            model_seconds_array = (data_mod['time'][p] - offset_in_minutes) * 60 + 43200

            model_data_array = data_mod['model'][p]
            obs_data_array = data_obs['obs'][p]
            clim_data_array = data_clim['clim'][p]
            clim_data_array = np.array(clim_data_array)

            # Handle model data:
            if model_data_array.ndim == 2:
                model_temp_slice = model_data_array[:, i]
            elif model_data_array.ndim == 1 and i == 0: # Is 1D, process only on first loop iteration
                model_temp_slice = model_data_array
            else: # Is 1D but not the first loop, or has unexpected dimensions
                print(f"    ! Skipping redundant/invalid model data for depth {depth_label}.")
                continue

            # Handle observation data:
            if obs_data_array.ndim == 2:
                obs_temp_slice = obs_data_array[:, i]
            elif obs_data_array.ndim <= 1 and obs_data_array.size > 0 and i == 0:
                obs_temp_slice = obs_data_array
            else:
                print(f"    ! Skipping redundant/invalid observation data for depth {depth_label}.")
                continue

            # Handle climatology data:
            if clim_data_array.ndim == 2:
                clim_temp_slice = clim_data_array[:, i]
                if debug:
                    print(f"clim_temp_slice: {clim_temp_slice}")
            elif clim_data_array.ndim <= 1 and clim_data_array.size > 0 and i == 0:
                clim_temp_slice = clim_data_array
                if debug:
                    print(f"clim_temp_slice: {clim_temp_slice}")
            else:
                clim_temp_slice = clim_data_array
                if debug:
                    print(f"clim_data_array.ndim: {clim_data_array.ndim}")
                    print(f"clim_data_array.size: {clim_data_array.size}")                
                    print(f"clim_temp_slice: {clim_temp_slice}")
            clim_temp_slice[clim_temp_slice > 9999] = np.nan
            
            # Create DataFrames
            d1 = pd.DataFrame({'time': model_seconds_array, 'model_temp': model_temp_slice})
            d2 = pd.DataFrame({'time': data_obs['time'][p], 'obs_temp': obs_temp_slice})
            d2.loc[d2['obs_temp'] >= 1.0e19, 'obs_temp'] = np.nan
            d3 = pd.DataFrame({'time': data_clim['time'][p], 'clim_temp': clim_temp_slice})
            
            # Merge and align data based on time
            data_frames = [d1, d2, d3]
            df_merged = reduce(lambda left, right: pd.merge(left, right, on='time', how='outer'), data_frames)
            df_merged = df_merged.sort_values(by='time').reset_index(drop=True)
            df_aligned = df_merged.dropna(subset=['model_temp', 'obs_temp'], how='all').copy()            

            if df_aligned.empty:
                print(f"    ! No valid data for depth {depth_label}. Skipping.")
                continue
            
            if debug:
                if platform_code == 'LION':
                    print(df_aligned['model_temp'])
            
            # --- STATISTICAL CALCULATIONS (VECTORIZED METHOD) ---
            diff = df_aligned['model_temp'] - df_aligned['obs_temp']
            bias = diff.mean()
            rms = np.sqrt((diff ** 2).mean())

            # --- PREPARATION FOR THE PLOT ---
            df_aligned['time'] = pd.to_datetime(df_aligned['time'], unit='s')
            start_day = df_aligned['time'].min().strftime('%Y-%m-%d')
            
            data_plot = df_aligned.rename(columns={'obs_temp': 'OBS', 'model_temp': 'MODEL', 'clim_temp': 'CLIM'})
            source = data_plot.melt(id_vars='time', value_vars=['OBS', 'MODEL', 'CLIM'], var_name="Temperature", value_name="y")
            
            y_min, y_max = source['y'].min(), source['y'].max()
            y_extension = [y_min, y_max]

            # The total height of your y-axis
            y_axis_range = y_extension[1] - y_extension[0]

            # Height of y-axis in the plot
            y_domain = [y_min - y_axis_range * 0.5, y_max + y_axis_range * 0.5]

            # The top of your plot's y-axis
            y_axis_top = y_domain[1]
            
            # Calculate an offset (e.g., 5% from the top)
            # You can adjust the 0.05 value to move the text
            offset = y_axis_range * 0.0000000000000010
           
            
            # --- ALTAIR CHART CREATION ---
            chart_title = alt.TitleParams(
                text=f'{platform_code} - {platform_name}',
                subtitle=f'Temperature [°C] at {depth_label} m',
                fontSize=18,
                subtitleFontSize=14,
                anchor='start'
            )

            axis_date = pd.date_range(start=start_day,periods=np.size(data_plot['MODEL']),freq="D")
            annotation_text = f"RMSE = {rms:.2f}\nbias = {bias:.3f}"
            annotation_df = pd.DataFrame({
                'x_pos': [axis_date[2]],
                'y_pos': [y_axis_top - offset],
                'text': [annotation_text]
            })
            
            text_layer = alt.Chart(annotation_df).mark_text(
                align='left', baseline='top', dx=5, dy=5, fontSize=12, lineBreak='\n'
            ).encode(x='x_pos:T', y='y_pos:Q', text='text:N')

            line_plot = alt.Chart(source).mark_line().encode(
                x=alt.X('time:T', title='Date', axis=alt.Axis(format="%d-%m-%Y", labelAngle=-45)),
                y=alt.Y('y:Q', title='Temperature [°C]', scale=alt.Scale(domain=y_domain)),
                color=alt.Color("Temperature:N",
                                scale=alt.Scale(
                                    domain=['OBS', 'MODEL', 'CLIM'],
                                    range=['#1f77b4', '#ff7f0e', '#2ca02c'] # Blue, Orange, Green
                    ))
            )

            nearest = alt.selection_point(nearest=True, on="pointerover", fields=["time"], empty=False)
            selectors = alt.Chart(source).mark_point().encode(x="time:T", opacity=alt.value(0)).add_params(nearest)
            points = line_plot.mark_point().encode(opacity=alt.condition(nearest, alt.value(1), alt.value(0)))
            text_tooltip = line_plot.mark_text(align="left", dx=5, dy=-5).encode(text=alt.condition(nearest, "y:Q", alt.value(" "), format=".2f"))
            rules = alt.Chart(source).mark_rule(color="gray").encode(x="time:T").transform_filter(nearest)

            final_layer = alt.layer(
                line_plot, selectors, points, rules, text_tooltip, text_layer
            ).properties(
                width=350, height=120, title=chart_title                
            ).interactive()
            
            charts_for_platform.append(final_layer)

        except Exception as e:
            print(f"    ERROR while processing depth {depth_label}: {e}")
            continue

    
    # --- ADDING MARKER AND POPUP TO FOLIUM ---
    if charts_for_platform:

        # This method creates a scrollable window inside the popup
        combined_chart = alt.vconcat(*charts_for_platform).resolve_scale(color='independent')
        chart_html_body = combined_chart.to_html()

        # 1. Create a new HTML structure with a centered div wrapper
        chart_html = f"""
        <!DOCTYPE html>
        <html>
        <head>
        </head>
        <body style="display: flex; justify-content: center;">
            {chart_html_body}
        </body>
        </html>
        """

        # 2. Create an IFrame with the chart HTML and set a fixed size for the viewport
        iframe = folium.IFrame(chart_html, width=600, height=450)

        # 3. Add the IFrame to a Popup
        popup = folium.Popup(iframe, max_width=600)

        icon = folium.features.CustomIcon(icon_url, icon_size=(28, 30))
        
        folium.Marker(
            location=[lat, lon],
            icon=icon,
            tooltip=platform_name,
            popup=popup
        ).add_to(m)

# --- MAP FINALIZATION ---
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite',
    overlay=False,
    control=True
).add_to(m)

# Display the map
m

#### 4. Salinity dataframe creation 
***

In order to visualize observations and model outputs, first it has to be created a dataframe containing needed information

In [ ]:
debug = False # Set to False to silence debug output

output_directory = parent_dir+'/output_NA/'

InsMerDir = output_directory+'OBSERVATION/output/statistic/dm/sea_water_practical_salinity' #merged/dm/sea_water_practical_salinity'
ModMerDir = output_directory+'MODEL/output/post-processed/dm/sea_water_practical_salinity' #merged/dm/sea_water_practical_salinity'
ClimMerDir = output_directory+'CLIM/output/statistic/dm/sea_water_practical_salinity' #merged/dm/sea_water_practical_salinity'


######################################
####### INSITU OBSERVATION ###########
######################################

pcode = []
pname = []
wcode = []
t = []
lon = []
lat = []
depth = []
obse = []
k_obs = []

mod_S = []
mod_time = []
k_mod = []

clim_S = []
clim_time = []
k_clim = []


sorted_files_obs = sorted([ x for x in os.listdir(InsMerDir) if x.endswith('nc')])
sorted_files_mod = sorted([ x for x in os.listdir(ModMerDir) if x.endswith('nc')])
sorted_files_clim = sorted([ x for x in os.listdir(ClimMerDir) if x.endswith('nc') and x.startswith(climdataset)])

for f in sorted_files_obs:
    mooring = f.split('_')[1]
    print(f"** Reading Platform: {mooring}")
    model_file = [mod_f for mod_f in sorted_files_mod if mod_f.split('_')[1] == mooring][0]

    fclim = 0
    try:
        clim_file = [clim_f for clim_f in sorted_files_clim if clim_f.split('_')[1] == mooring][0]
    except IndexError as e:
        msg = f"Climatological salinity file not found for platform {mooring}"
        print(msg)
    else:
        msg = f"Climatological salinity file ** FOUND ** for platform {mooring}"
        print(msg)
        fclim = 1        
       
    
    # Read insitu netcdf
    i_nc = Dataset(InsMerDir+'/'+f,'r')
    print(f"Insitu file: {InsMerDir+'/'+f}")
    t.append(i_nc.variables['time'][:])
    lon.append(i_nc.variables['lon'][:])
    lat.append(i_nc.variables['lat'][:])
    depth.append(i_nc.variables['depth'][:])
    obse.append(np.squeeze(i_nc.variables['sea_water_practical_salinity']))
    k_obs.append('insitu')
    pcode.append(i_nc.getncattr('platform_code'))

    # For some probes platform_name is not present or empty.
    # In this cases use platform_code instead
    final_value = None # Initialize a variable for the final value

    # 1. Try to use 'platform_name'
    if hasattr(i_nc, 'platform_name'):
        value = i_nc.getncattr('platform_name')
        # Check that the value is not empty or just whitespace
        if value and str(value).strip():
            final_value = value

    # 2. If no value was found, try with 'platform_code'
    if not final_value and hasattr(i_nc, 'platform_code'):
        code_value = i_nc.getncattr('platform_code')
        # Check here as well that the value is not empty
        if code_value and str(code_value).strip():
            final_value = code_value

    # 3. Finally, append the value to the list only if a valid one was found
    if final_value:
        pname.append(final_value)

    wcode.append(i_nc.getncattr('wmo_platform_code'))

    
#     # ######################################
#     # #######        MODEL       ###########
#     # ######################################

    # Read model netcdf
    m_nc = Dataset(ModMerDir+'/'+model_file,'r')
    print(f"Model file: {ModMerDir+'/'+model_file}")
    mod_time.append(m_nc.variables['time'][:])
    mod_S.append(np.squeeze(m_nc.variables['so']))
    k_mod.append('model')


#     # ######################################
#     # #########   CLIMATOLOGY    ###########
#     # ######################################

    # Read climatology netcdf
    if fclim == 1:
        c_nc = Dataset(ClimMerDir+'/'+clim_file,'r')
        print(f"Climatology file: {ClimMerDir+'/'+clim_file}")
        clim_time.append(c_nc.variables['time'][:])
        clim_S.append(np.squeeze(c_nc.variables['sea_water_practical_salinity']))
        k_clim.append('climatology')
    elif fclim == 0:
        offset_in_minutes = (datetime.datetime(1970, 1, 1) - datetime.datetime(1900, 1, 1)).total_seconds() / 60
        clim_time_seconds = (m_nc.variables['time'][:] - offset_in_minutes) * 60 + 43200
        clim_time.append(clim_time_seconds)
        clim_S.append([np.nan] * len((m_nc.variables['time'][:])))
        k_clim.append('climatology')

# end of the loop on sorted_files_obs

# Create dataframes
data_obs = pd.DataFrame({'platform_code': pcode,
        'platform_name': pname,
        'wmo_platform_code': wcode,
        'time': t,
        'lon': lon,
        'lat': lat,
        'depth': depth,
        'obs': obse,
        'kind': k_obs
        })

data_mod = pd.DataFrame({'time': mod_time,'model': mod_S,'kind': k_mod})

data_clim = pd.DataFrame({'time': clim_time,'clim': clim_S,'kind': k_clim})

if debug:
    display(data_clim)  # or display(data_obs) or display(data_mod)

#### 5. Visualization of salinity data and computation of model skill scores
***

Salinity time series visualization, RMSE and bias computation

In [ ]:
# --- INITIAL SETUP ---
debug = False  # Set to True to enable debug output
m = folium.Map(location=[40.5, 12.5], zoom_start=6)
icon_url = parent_dir + '/ICON/buoy_icon_2.png'

# --- MAIN LOOP OVER PLATFORMS ---
for p in np.arange(0,data_obs.shape[0]):    
    print(f"** Processing Platform: {data_obs['platform_code'][p]}")
    
    charts_for_platform = []
    lat = data_obs['lat'][p]
    lon = data_obs['lon'][p]
    platform_code = str(data_obs['platform_code'][p])
    # Modify platform_name for the probes where platform_name is not present or empty.
    platform_name = str(data_obs['platform_name'][p])
    if platform_name == '68422':
        platform_name = 'Pylos'
    elif platform_name == '61277':
        platform_name = 'E1M3A'
    elif platform_name == 'HERAKLION':
        platform_name = 'Heraklion'

    # --- INNER LOOP OVER DEPTHS ---
    for i, depth_level in enumerate(data_obs['depth'][p]):
        depth_label = str(np.around(depth_level,2))
        print(f"  - Processing depth: {depth_label} m")

        try:
            # --- DATA PREPARATION FOR THE CURRENT DEPTH ---
            offset_in_minutes = (datetime.datetime(1970, 1, 1) - datetime.datetime(1900, 1, 1)).total_seconds() / 60
            model_seconds_array = (data_mod['time'][p] - offset_in_minutes) * 60 + 43200

            model_data_array = data_mod['model'][p]
            obs_data_array = data_obs['obs'][p]
            clim_data_array = data_clim['clim'][p]
            clim_data_array = np.array(clim_data_array)

            # Handle model data:
            if model_data_array.ndim == 2:
                model_temp_slice = model_data_array[:, i]
            elif model_data_array.ndim == 1 and i == 0: # Is 1D, process only on first loop iteration
                model_temp_slice = model_data_array
            else: # Is 1D but not the first loop, or has unexpected dimensions
                print(f"    ! Skipping redundant/invalid model data for depth {depth_label}.")
                continue

            # Handle observation data:
            if obs_data_array.ndim == 2:
                obs_temp_slice = obs_data_array[:, i]
            elif obs_data_array.ndim <= 1 and obs_data_array.size > 0 and i == 0:
                obs_temp_slice = obs_data_array
            else:
                print(f"    ! Skipping redundant/invalid observation data for depth {depth_label}.")
                continue

            # Handle climatology data:
            if clim_data_array.ndim == 2:
                clim_temp_slice = clim_data_array[:, i]
                if debug:
                    print(f"clim_temp_slice: {clim_temp_slice}")
            elif clim_data_array.ndim <= 1 and clim_data_array.size > 0 and i == 0:
                clim_temp_slice = clim_data_array
                if debug:
                    print(f"clim_temp_slice: {clim_temp_slice}")
            else:
                clim_temp_slice = clim_data_array
                if debug:
                    print(f"clim_data_array.ndim: {clim_data_array.ndim}")
                    print(f"clim_data_array.size: {clim_data_array.size}")                
                    print(f"clim_temp_slice: {clim_temp_slice}")
            clim_temp_slice[clim_temp_slice > 9999] = np.nan         

            
            # Create DataFrames
            d1 = pd.DataFrame({'time': model_seconds_array, 'model_temp': model_temp_slice})
            d2 = pd.DataFrame({'time': data_obs['time'][p], 'obs_temp': obs_temp_slice})
            d2.loc[d2['obs_temp'] >= 1.0e19, 'obs_temp'] = np.nan
            d3 = pd.DataFrame({'time': data_clim['time'][p], 'clim_temp': clim_temp_slice})
            
            # Merge and align data based on time
            data_frames = [d1, d2, d3]
            df_merged = reduce(lambda left, right: pd.merge(left, right, on='time', how='outer'), data_frames)
            df_merged = df_merged.sort_values(by='time').reset_index(drop=True)
            df_aligned = df_merged.dropna(subset=['model_temp', 'obs_temp'], how='all').copy()            

            if df_aligned.empty:
                print(f"    ! No valid data for depth {depth_label}. Skipping.")
                continue

            # --- STATISTICAL CALCULATIONS (VECTORIZED METHOD) ---
            diff = df_aligned['model_temp'] - df_aligned['obs_temp']
            bias = diff.mean()
            rms = np.sqrt((diff ** 2).mean())

            # --- PREPARATION FOR THE PLOT ---
            df_aligned['time'] = pd.to_datetime(df_aligned['time'], unit='s')
            start_day = df_aligned['time'].min().strftime('%Y-%m-%d')
            
            data_plot = df_aligned.rename(columns={'obs_temp': 'OBS', 'model_temp': 'MODEL', 'clim_temp': 'CLIM'})
            source = data_plot.melt(id_vars='time', value_vars=['OBS', 'MODEL', 'CLIM'], var_name="Salinity", value_name="y")
            
            y_min, y_max = source['y'].min(), source['y'].max()
            y_extension = [y_min, y_max]

            # The total height of your y-axis
            y_axis_range = y_extension[1] - y_extension[0]

            # Height of y-axis in the plot
            y_domain = [y_min - y_axis_range * 0.5, y_max + y_axis_range * 0.5]

            # The top of your plot's y-axis
            y_axis_top = y_domain[1]
            
            # Calculate an offset (e.g., 5% from the top)
            # You can adjust the 0.05 value to move the text
            offset = y_axis_range * 0.0000000000000010
           
            
            # --- ALTAIR CHART CREATION ---
            chart_title = alt.TitleParams(
                text=f'{platform_code} - {platform_name}',
                subtitle=f'Salinity [PSU] at {depth_label} m',
                fontSize=18,
                subtitleFontSize=14,
                anchor='start'
            )

            axis_date = pd.date_range(start=start_day,periods=np.size(data_plot['MODEL']),freq="D")
            annotation_text = f"RMSE = {rms:.2f}\nbias = {bias:.3f}"
            annotation_df = pd.DataFrame({
                'x_pos': [axis_date[2]],
                'y_pos': [y_axis_top - offset],
                'text': [annotation_text]
            })
            
            text_layer = alt.Chart(annotation_df).mark_text(
                align='left', baseline='top', dx=5, dy=5, fontSize=12, lineBreak='\n'
            ).encode(x='x_pos:T', y='y_pos:Q', text='text:N')

            line_plot = alt.Chart(source).mark_line().encode(
                x=alt.X('time:T', title='Date', axis=alt.Axis(format="%d-%m-%Y", labelAngle=-45)),
                y=alt.Y('y:Q', title='Salinity [PSU]', scale=alt.Scale(domain=y_domain)),
                color=alt.Color("Salinity:N",
                                scale=alt.Scale(
                                    domain=['OBS', 'MODEL', 'CLIM'],
                                    range=['#1f77b4', '#ff7f0e', '#2ca02c'] # Blue, Orange, Green
                    ))
            )

            nearest = alt.selection_point(nearest=True, on="pointerover", fields=["time"], empty=False)
            selectors = alt.Chart(source).mark_point().encode(x="time:T", opacity=alt.value(0)).add_params(nearest)
            points = line_plot.mark_point().encode(opacity=alt.condition(nearest, alt.value(1), alt.value(0)))
            text_tooltip = line_plot.mark_text(align="left", dx=5, dy=-5).encode(text=alt.condition(nearest, "y:Q", alt.value(" "), format=".2f"))
            rules = alt.Chart(source).mark_rule(color="gray").encode(x="time:T").transform_filter(nearest)

            final_layer = alt.layer(
                line_plot, selectors, points, rules, text_tooltip, text_layer
            ).properties(
                width=350, height=120, title=chart_title                
            ).interactive()
            
            charts_for_platform.append(final_layer)

        except Exception as e:
            print(f"    ERROR while processing depth {depth_label}: {e}")
            continue

    
    # --- ADDING MARKER AND POPUP TO FOLIUM ---
    if charts_for_platform:

        # This method creates a scrollable window inside the popup
        combined_chart = alt.vconcat(*charts_for_platform).resolve_scale(color='independent')
        chart_html_body = combined_chart.to_html()

        # 1. Create a new HTML structure with a centered div wrapper
        chart_html = f"""
        <!DOCTYPE html>
        <html>
        <head>
        </head>
        <body style="display: flex; justify-content: center;">
            {chart_html_body}
        </body>
        </html>
        """

        # 2. Create an IFrame with the chart HTML and set a fixed size for the viewport
        iframe = folium.IFrame(chart_html, width=600, height=450)

        # 3. Add the IFrame to a Popup
        popup = folium.Popup(iframe, max_width=600)

        icon = folium.features.CustomIcon(icon_url, icon_size=(28, 30))
        
        folium.Marker(
            location=[lat, lon],
            icon=icon,
            tooltip=platform_name,
            popup=popup
        ).add_to(m)

# --- MAP FINALIZATION ---
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite',
    overlay=False,
    control=True
).add_to(m)

# Display the map
m